# SurfaceTarget-CRC — phase 02 — patch 2 1

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 2 — PATCH 2.1
#
# CPTAC COLON PROTEOMICS ACQUISITION AND AUDIT
#
# Objectives:
# 1. Download processed CPTAC colon proteomics datasets
# 2. Verify downloaded file structure
# 3. Record file integrity checksums
# 4. Load CPTAC clinical annotations
# 5. Identify published MSS and MSI-H classifications
# 6. Assess protein measurement availability
# 7. Evaluate transcriptomic candidate coverage
# 8. Save reproducible source and audit metadata
#
# IMPORTANT:
# - No proteomic differential expression yet
# - No protein abundance imputation
# - No merging of TCGA and CPTAC patient IDs
# - Missing protein measurements are not zeros
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests

import hashlib
import json
import time
import os


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"

RAW = ROOT / "data/raw/cptac_colon"

RESULTS = ROOT / "results/tables"

for directory in [META, RAW, RESULTS]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

CANDIDATE_PATH = (
    RESULTS /
    "tcga_coad_paired_support_comparison.csv"
)

print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.1")
print("CPTAC COLON PROTEOMICS DATA ACQUISITION")
print("=" * 65)


# ------------------------------------------------------------
# 2. Define source datasets
# ------------------------------------------------------------

# Source:
# CPTAC Prospective Colon Cancer Study
# NCI Proteomic Data Commons: PDC000116
#
# Processed matrices distributed through LinkedOmics.
#
# Tumor and normal unshared TMT log-ratio matrices
# are retained separately.
#
# The paired tumor-normal matrix contains
# reported within-patient protein log2 ratios.
#
# We will verify cohort membership and
# measurement availability before analysis.

BASE_URL = (
    "https://linkedomics.org/cptac-colon/"
)

SOURCE_FILES = {

    "tumor_proteomics": (
        "Human__CPTAC_COAD__PNNL__Proteome__TMT__"
        "03_01_2017__BCM__Gene__"
        "PNNL_Tumor_TMT_UnsharedLogRatio.cct"
    ),

    "normal_proteomics": (
        "Human__CPTAC_COAD__PNNL__Proteome__TMT__"
        "03_01_2017__BCM__Gene__"
        "PNNL_Normal_TMT_UnsharedLogRatio.cct"
    ),

    "paired_proteomics": (
        "Human__CPTAC_COAD__PNNL__Proteome__TMT__"
        "03_01_2017__BCM__Gene__"
        "Tumor_Normal_log2FC.cct"
    ),

    "clinical": (
        "Human__CPTAC_COAD__MS__Clinical__Clinical__"
        "03_01_2017__CPTAC__Clinical__BCM.tsi"
    )
}

SOURCE_URLS = {
    name: BASE_URL + filename
    for name, filename in SOURCE_FILES.items()
}

print("\nStudy: CPTAC Prospective Colon Cancer")
print("PDC study identifier: PDC000116")
print("Processed-data source: LinkedOmics")
print("Datasets requested:", len(SOURCE_FILES))


# ------------------------------------------------------------
# 3. Create download session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "User-Agent": "SurfaceTarget-CRC-Research/1.0",
    "Accept": "*/*"
})


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


def valid_matrix_header(path):

    if not path.exists():
        return False

    if path.stat().st_size < 1000:
        return False

    with open(
        path,
        "rb"
    ) as f:

        first_line = f.readline(
            1024 * 1024
        )

    return first_line.startswith(
        b"attrib_name\t"
    )


def download_dataset(
    url,
    destination,
    max_attempts=3
):

    if valid_matrix_header(destination):

        print("Previously downloaded file found.")

        return "Existing file"

    temporary = Path(
        str(destination) + ".part"
    )

    for attempt in range(
        1,
        max_attempts + 1
    ):

        try:

            if temporary.exists():
                temporary.unlink()

            with session.get(
                url,
                stream=True,
                timeout=(30, 120)
            ) as response:

                response.raise_for_status()

                downloaded_bytes = 0

                with open(
                    temporary,
                    "wb"
                ) as output:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if chunk:

                            output.write(chunk)

                            downloaded_bytes += len(chunk)

                            if downloaded_bytes > 40 * 1024**2:
                                raise RuntimeError(
                                    "Unexpectedly large dataset."
                                )

            if not valid_matrix_header(temporary):

                raise RuntimeError(
                    "Downloaded file is not a valid "
                    "LinkedOmics matrix."
                )

            os.replace(
                temporary,
                destination
            )

            return "Downloaded"

        except Exception as error:

            print(
                f"Attempt {attempt} failed: {error}"
            )

            if temporary.exists():
                temporary.unlink()

            if attempt == max_attempts:
                raise

            time.sleep(
                2 ** attempt
            )


# ------------------------------------------------------------
# 4. Download source files
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DOWNLOADING CPTAC DATASETS")
print("-" * 65)

source_records = []

for name, filename in SOURCE_FILES.items():

    print("\nDataset:", name)

    destination = RAW / filename

    status = download_dataset(
        SOURCE_URLS[name],
        destination
    )

    source_records.append({

        "dataset": name,

        "filename": filename,

        "source_url": SOURCE_URLS[name],

        "local_path": str(destination),

        "download_status": status,

        "file_size_bytes": destination.stat().st_size,

        "sha256": sha256_file(destination)
    })

    print("Status:", status)

    print(
        "Size:",
        round(
            destination.stat().st_size / 1024**2,
            2
        ),
        "MiB"
    )

print("\nAll source files acquired.")


# ------------------------------------------------------------
# 5. Load quantitative proteomics matrices
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING PROTEOMICS MATRICES")
print("-" * 65)


def load_proteomics(name):

    path = RAW / SOURCE_FILES[name]

    raw = pd.read_csv(
        path,
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    raw.index = (
        raw.index.astype(str).str.strip()
    )

    raw.columns = (
        raw.columns.astype(str).str.strip()
    )

    if raw.index.isna().any():
        raise RuntimeError(
            f"Missing protein identifiers: {name}"
        )

    if raw.columns.duplicated().any():
        raise RuntimeError(
            f"Duplicate sample identifiers: {name}"
        )

    # Convert to numeric while detecting
    # unexpected nonnumeric entries.

    numeric = raw.apply(
        pd.to_numeric,
        errors="coerce"
    )

    invalid = (
        raw.notna() & numeric.isna()
    )

    if invalid.to_numpy().any():

        raise RuntimeError(
            f"Unexpected nonnumeric values: {name}"
        )

    if numeric.shape[0] < 3000:
        raise RuntimeError(
            f"Unexpectedly few proteins: {name}"
        )

    if numeric.shape[1] < 50:
        raise RuntimeError(
            f"Unexpectedly few samples: {name}"
        )

    return numeric


proteomics = {

    "tumor_proteomics":
        load_proteomics("tumor_proteomics"),

    "normal_proteomics":
        load_proteomics("normal_proteomics"),

    "paired_proteomics":
        load_proteomics("paired_proteomics")
}

for name, matrix in proteomics.items():

    print("\nDataset:", name)

    print(
        "Dimensions:",
        matrix.shape
    )

    print(
        "Unique gene symbols:",
        matrix.index.nunique()
    )

    print(
        "Duplicated gene-symbol rows:",
        int(matrix.index.duplicated().sum())
    )

    print(
        "Missing measurement percentage:",
        round(
            100 * matrix.isna().to_numpy().mean(),
            2
        )
    )


# ------------------------------------------------------------
# 6. Load CPTAC clinical metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LOADING CPTAC CLINICAL METADATA")
print("-" * 65)

clinical_path = (
    RAW / SOURCE_FILES["clinical"]
)

clinical_raw = pd.read_csv(
    clinical_path,
    sep="\t",
    index_col=0,
    na_values=["NA", "N/A", ""],
    low_memory=False
)

# LinkedOmics clinical file orientation:
#
# Rows = clinical attributes
# Columns = patients
#
# Transpose to patients x attributes.

clinical = clinical_raw.T.copy()

clinical.index = (
    clinical.index.astype(str).str.strip()
)

clinical.index.name = "cptac_patient_id"

if clinical.index.duplicated().any():

    raise RuntimeError(
        "Duplicate CPTAC clinical patient IDs."
    )

if "Mutation_Phenotype" not in clinical.columns:

    raise RuntimeError(
        "Published MSI classification "
        "not found in CPTAC clinical metadata."
    )

clinical["Mutation_Phenotype"] = (
    clinical["Mutation_Phenotype"]
    .astype("string")
    .str.strip()
    .str.upper()
)

allowed_statuses = {
    "MSS",
    "MSI-H"
}

observed_statuses = set(
    clinical["Mutation_Phenotype"].dropna()
)

unexpected = (
    observed_statuses - allowed_statuses
)

if unexpected:

    print(
        "WARNING: Additional phenotype labels:",
        unexpected
    )

print("\nClinical patients:", len(clinical))

print("\nPublished mutation phenotype:")

print(
    clinical["Mutation_Phenotype"]
    .fillna("NOT AVAILABLE")
    .value_counts()
    .to_string()
)


# ------------------------------------------------------------
# 7. Validate sample correspondence
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEOMICS / CLINICAL SAMPLE CORRESPONDENCE")
print("-" * 65)

tumor_ids = set(
    proteomics["tumor_proteomics"].columns
)

normal_ids = set(
    proteomics["normal_proteomics"].columns
)

paired_ids = set(
    proteomics["paired_proteomics"].columns
)

clinical_ids = set(
    clinical.index
)

print("\nTumor proteomics patients:", len(tumor_ids))

print("Normal proteomics patients:", len(normal_ids))

print("Paired ratio patients:", len(paired_ids))

print("Clinical patients:", len(clinical_ids))

print(
    "\nTumor/normal identifier overlap:",
    len(tumor_ids & normal_ids)
)

print(
    "Paired ratio/clinical overlap:",
    len(paired_ids & clinical_ids)
)

print(
    "All three protein matrices overlap:",
    len(tumor_ids & normal_ids & paired_ids)
)

# Sample correspondence is audited,
# not assumed to be complete.


# ------------------------------------------------------------
# 8. Identify CPTAC MSS samples
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CPTAC MSS COHORT IDENTIFICATION")
print("-" * 65)

paired = proteomics["paired_proteomics"]

paired_metadata = clinical.reindex(
    paired.columns
).copy()

paired_metadata["clinical_match"] = (
    paired_metadata.index.isin(
        clinical.index
    )
)

paired_metadata["msi_classification"] = (
    paired_metadata["Mutation_Phenotype"]
    .fillna("NOT AVAILABLE")
)

paired_distribution = (
    paired_metadata["msi_classification"]
    .value_counts()
)

print("\nPaired proteomics by published MSI status:")

print(
    paired_distribution.to_string()
)

mss_ids = (
    paired_metadata.index[
        paired_metadata[
            "msi_classification"
        ].eq("MSS")
    ].tolist()
)

msih_ids = (
    paired_metadata.index[
        paired_metadata[
            "msi_classification"
        ].eq("MSI-H")
    ].tolist()
)

print("\nPublished MSS paired patients:", len(mss_ids))

print("Published MSI-H paired patients:", len(msih_ids))

if len(mss_ids) < 30:

    raise RuntimeError(
        "Unexpectedly few published MSS "
        "samples in paired proteomics data."
    )

print("\nCPTAC MSS cohort identification: PASSED")


# ------------------------------------------------------------
# 9. Validate transcriptomic screening genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TRANSCRIPTOMIC CANDIDATE COVERAGE")
print("-" * 65)

if not CANDIDATE_PATH.exists():

    raise FileNotFoundError(
        "Patch 1.22 candidate comparison "
        "table not found."
    )

candidates = pd.read_csv(
    CANDIDATE_PATH,
    dtype={"gene_id": str},
    low_memory=False
)

required_candidate_columns = {
    "gene_id",
    "gene_name",
    "combined_transcriptomic_support"
}

if not required_candidate_columns.issubset(
    candidates.columns
):

    raise RuntimeError(
        "Required screening candidate "
        "columns are missing."
    )

supported_mask = (
    candidates["combined_transcriptomic_support"]
    .astype(str)
    .str.lower()
    .eq("true")
)

supported = candidates[
    supported_mask
].copy()

print(
    "\nTranscriptomic screening genes:",
    len(supported)
)

if len(supported) != 948:

    raise RuntimeError(
        "Transcriptomic screening gene count "
        "differs from Patch 1.22."
    )

supported["gene_name"] = (
    supported["gene_name"]
    .astype(str)
    .str.strip()
)

# Coverage counts report whether a gene
# symbol appears in the protein assay.
#
# They do not establish protein enrichment.

paired_gene_counts = (
    paired.index.value_counts()
)

tumor_gene_set = set(
    proteomics["tumor_proteomics"].index
)

normal_gene_set = set(
    proteomics["normal_proteomics"].index
)

paired_gene_set = set(
    paired.index
)

supported["in_tumor_proteomics"] = (
    supported["gene_name"].isin(
        tumor_gene_set
    )
)

supported["in_normal_proteomics"] = (
    supported["gene_name"].isin(
        normal_gene_set
    )
)

supported["in_paired_proteomics"] = (
    supported["gene_name"].isin(
        paired_gene_set
    )
)

supported["paired_symbol_row_count"] = (
    supported["gene_name"]
    .map(paired_gene_counts)
    .fillna(0)
    .astype(int)
)

# We deliberately do not collapse duplicate
# protein identifiers in this acquisition patch.

print("\nProtein assay coverage:")

for column in [
    "in_tumor_proteomics",
    "in_normal_proteomics",
    "in_paired_proteomics"
]:

    print(
        column,
        ":",
        int(supported[column].sum()),
        "/",
        len(supported)
    )

print(
    "\nCandidates with ambiguous duplicate "
    "paired protein-symbol rows:",
    int(
        (
            supported["paired_symbol_row_count"] > 1
        ).sum()
    )
)


# ------------------------------------------------------------
# 10. Inspect leading transcriptomic genes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING GENE PROTEOMICS COVERAGE")
print("-" * 65)

top_genes = [
    "CDH3",
    "CLDN1",
    "KRT80",
    "ETV4",
    "FOXQ1",
    "ESM1",
    "CEMIP",
    "INHBA",
    "SIM2",
    "WNT2"
]

top_coverage = supported[
    supported["gene_name"].isin(
        top_genes
    )
].copy()

top_coverage["gene_name"] = pd.Categorical(
    top_coverage["gene_name"],
    categories=top_genes,
    ordered=True
)

top_coverage = (
    top_coverage
    .sort_values("gene_name")
)

print("\n")

print(
    top_coverage[
        [
            "gene_name",
            "in_tumor_proteomics",
            "in_normal_proteomics",
            "in_paired_proteomics",
            "paired_symbol_row_count"
        ]
    ].to_string(index=False)
)

print("\nIMPORTANT:")
print(
    "Protein measurement availability "
    "is not evidence of tumor enrichment."
)


# ------------------------------------------------------------
# 11. Save CPTAC dataset audit
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING CPTAC AUDIT RESULTS")
print("-" * 65)

source_manifest = pd.DataFrame(
    source_records
)

source_manifest.to_csv(
    META / "cptac_colon_source_manifest.csv",
    index=False
)

clinical.reset_index().to_csv(
    META / "cptac_colon_clinical_metadata.csv",
    index=False
)

paired_metadata.reset_index().to_csv(
    META / "cptac_colon_paired_sample_metadata.csv",
    index=False
)

supported.to_csv(
    RESULTS / "cptac_transcriptomic_candidate_coverage.csv",
    index=False
)

top_coverage.to_csv(
    RESULTS / "cptac_top_gene_protein_coverage.csv",
    index=False
)

inventory_records = []

for name, matrix in proteomics.items():

    inventory_records.append({

        "dataset": name,

        "n_gene_rows": len(matrix),

        "n_unique_gene_symbols":
            matrix.index.nunique(),

        "n_patient_columns":
            len(matrix.columns),

        "missing_measurements": int(
            matrix.isna().to_numpy().sum()
        ),

        "missing_percent": float(
            100 * matrix.isna().to_numpy().mean()
        ),

        "duplicate_gene_symbol_rows": int(
            matrix.index.duplicated().sum()
        )
    })

inventory_df = pd.DataFrame(
    inventory_records
)

inventory_df.to_csv(
    RESULTS / "cptac_colon_proteomics_inventory.csv",
    index=False
)

print("\nCPTAC metadata and audit tables saved.")


# ------------------------------------------------------------
# 12. Save reproducibility metadata
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.1",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study": "CPTAC Prospective Colon Cancer",

    "pdc_study_id": "PDC000116",

    "processed_data_source": "LinkedOmics",

    "dataset_sources": SOURCE_URLS,

    "source_file_sha256": {
        record["dataset"]: record["sha256"]
        for record in source_records
    },

    "protein_measurement_type": (
        "TMT-based relative protein abundance "
        "and reported matched tumor-normal log2 ratios"
    ),

    "paired_proteomics_patients": int(
        len(paired_ids)
    ),

    "published_mss_paired_patients": int(
        len(mss_ids)
    ),

    "published_msih_paired_patients": int(
        len(msih_ids)
    ),

    "transcriptomic_screening_genes": int(
        len(supported)
    ),

    "protein_differential_expression_performed": False,

    "protein_imputation_performed": False,

    "tcga_cptac_patient_level_merging": False,

    "limitations": [
        "CPTAC prospective colon and TCGA are separate cohorts.",
        "Published MSS annotations must be retained.",
        "TMT protein measurements are relative abundances.",
        "Missing protein measurements are not biological zeros.",
        "Gene-symbol mapping can be ambiguous.",
        "Protein quantification does not prove surface accessibility.",
        "No tumor-enrichment statistics calculated yet.",
        "No therapeutic target has been validated."
    ]
}

with open(
    META / "cptac_colon_acquisition_provenance.json",
    "w"
) as output:

    json.dump(
        provenance,
        output,
        indent=4
    )


# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.1 SUMMARY")
print("=" * 65)

print("\nCPTAC source datasets:", len(source_records))

print(
    "Paired proteomics patients:",
    len(paired_ids)
)

print(
    "Published MSS paired patients:",
    len(mss_ids)
)

print(
    "Published MSI-H paired patients:",
    len(msih_ids)
)

print(
    "\nTranscriptomic screening genes:",
    len(supported)
)

print(
    "Screening genes represented "
    "in paired protein assay:",
    int(
        supported["in_paired_proteomics"].sum()
    )
)

print("\nSource files stored in:")
print(RAW)

print("\nAll acquisition metadata saved.")

print("\nNo protein differential expression performed.")

print("No source measurements modified.")

print("No TCGA/CPTAC patient IDs merged.")

print("\nPATCH 2.1 COMPLETE")
print("=" * 65)